In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
pip install -U langchain-community sentence-transformers

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_9157/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
!pip install "unstructured[all-docs]" unstructured-inference unstructured.pytesseract pillow opencv-python-headless pdf2image pdfminer.six pymupdf pytesseract

In [ ]:
 # load and extract images, tables, and chunk text
from unstructured.partition.pdf import partition_pdf

filename = "/content/Olympic_for_Indian_National.pdf"

pdf_elements = partition_pdf(
    filename=filename,
    extract_images_in_pdf=True,
    strategy = "hi_res",
    hi_res_model_name="yolox",
    infer_table_structure=True,
    chunking_strategy="by_title",
    max_characters=3000,
    combine_text_under_n_chars=200,
)

In [ ]:
from collections import Counter
category_counts = Counter(str(type(element)) for element in pdf_elements)
unique_categories = set(category_counts)
category_counts

Counter({"<class 'unstructured.documents.elements.CompositeElement'>": 5})

In [ ]:
unique_types = {el.to_dict()['type'] for el in pdf_elements}
unique_types

{'CompositeElement'}

In [ ]:
!pip install langchain langchain-core langchain-community

In [ ]:
from langchain_core.documents import Document
documents = [Document(page_content=el.text, metadata={"source": filename}) for el in pdf_elements]

In [ ]:
pip install faiss-cpu

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(documents, embeddings)


In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
pip install langchain-groq

In [ ]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq



from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

template = """
You are a helpful assistant that answers questions based on the provided context.
Use the provided context to answer the question.

Question: {input}
Context: {context}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)



retriever = vectorstore.as_retriever(search_kwargs={"k": 3})



rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)




In [ ]:
from IPython.display import display, Markdown

response = rag_chain.invoke("can u give any 1 indian footballer")
display(Markdown(response))


Based on the provided context, one Indian footballer from Assam is:

1. Dr. Talimeren Ao (also known as T Ao) 
2. Gilbertson Sangma 
3. Arup Das 
4. Syed Abid Imam 
5. Holicharan Narzary 
6. Vinit Rai